<a href="https://colab.research.google.com/github/Lucas-Granucci/MULTI-NER/blob/main/MULTI_NER.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# !pip install datasets pytorch-crf

# **Machine learning for low-resource NLP**: Advancing AI for Linguistic Inclusion
Cross-lingual transfer learning and pseudo-labeling for multilingual named entity recognition

To connect to local runtime:

```
docker run --gpus=all -p 127.0.0.1:9000:8080 us-docker.pkg.dev/colab-images/public/runtime
```



**General Imports:** Import fundamental libraries

In [ ]:
import gc
import torch
import random
import numpy as np
import pandas as pd

**Constants:** Define constants for entire project

In [ ]:
# Miscelanous
RANDOM_STATE          = 42
DEVICE                = torch.device("cuda")

# Data
low_resource_langs    = ["mg", "fo", "co", "hsb", "bh", "cv"]
high_resource_langs   = ["id", "is", "it", "pl", "hi", "tt"]

NUM_TAGS              = 7
BATCH_SIZE            = 32
MAX_SEQ_LEN           = 128

# Training
EPOCHS                = 20
PATIENCE              = 5
BERT_LEARNING_RATE    = 0.00003
LSTM_LEARNING_RATE    = 0.005
CRF_LEARNING_RATE     = 0.00005
WEIGHT_DECAY          = 0.01

**Set Random Seed:** Ensure random seeds are all set to esnure reproducibility of results

In [ ]:
def set_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    np.random.seed(seed)
    random.seed(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(RANDOM_STATE)

**Data Processing:** Load WikiANN data from HuggingFace and split into train/val/test

In [ ]:
from datasets import load_dataset
from sklearn.model_selection import train_test_split

def load_wikiann_datasets(language_codes, cutoff = None):

    language_data = {}
    for lang in language_codes:
        # Load raw data from hugging face
        lang_dataset = load_dataset("unimelb-nlp/wikiann", name=lang)

        # Get data from different splits and combine
        train_df = pd.DataFrame(lang_dataset["train"])
        val_df = pd.DataFrame(lang_dataset["validation"])
        test_df = pd.DataFrame(lang_dataset["test"])

        complete_df = pd.concat([train_df, val_df, test_df]).reset_index(drop=True)
        complete_df = complete_df.head(cutoff) if cutoff else complete_df

        # Split data into new train/val/test splits
        train, temp = train_test_split(complete_df, test_size = 0.2, random_state = RANDOM_STATE)
        val, test = train_test_split(temp, test_size = 0.5, random_state = RANDOM_STATE)

        language_data[lang] = {"train": train, "val": val, "test": test}

    return language_data

# Download and store data
low_resource_datasets = load_wikiann_datasets(low_resource_langs)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/158k [00:00<?, ?B/s]

validation-00000-of-00001.parquet:   0%|          | 0.00/9.00k [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/9.07k [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/8.91k [00:00<?, ?B/s]

...

test-00000-of-00001.parquet:   0%|          | 0.00/10.8k [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/11.2k [00:00<?, ?B/s]

Generating validation split:   0%|          | 0/100 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/100 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/100 [00:00<?, ? examples/s]

**NER Dataset:** Define dataset for storing and processing NER data

In [ ]:
from transformers import BertTokenizerFast

class NERDataset:
    def __init__(self, texts, tags):
        self.texts = texts
        self.tags = tags

        self.tokenizer = BertTokenizerFast.from_pretrained(
            "google-bert/bert-base-multilingual-cased", do_lower_case = True
        )

        self.CLS_TOKEN = [101]
        self.SEP_TOKEN = [102]
        self.PAD_TOKEN = [0]
        self.MAX_LEN = MAX_SEQ_LEN

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, index):
        text = self.texts[index]
        tags = self.tags[index]

        token_ids = []
        target_tags = []
        for i, word in enumerate(text):
            word_ids = self.tokenizer.encode(word, add_special_tokens = False)
            token_ids.extend(word_ids)
            target_tags.extend(len(word_ids) * [tags[i]])

        # Resize for special tokens
        token_ids = token_ids[:self.MAX_LEN - 2]
        target_tags = target_tags[:self.MAX_LEN - 2]

        # Add special tokens
        token_ids = self.CLS_TOKEN + token_ids + self.SEP_TOKEN
        target_tags = self.PAD_TOKEN + target_tags + self.PAD_TOKEN

        attention_mask = [1] * len(token_ids)
        token_type_ids = [0] * len(token_ids)

        # Add padding to make sure all inputs are the same size
        padding_len = self.MAX_LEN - len(token_ids)
        token_ids += [0] * padding_len
        target_tags += [0] * padding_len
        attention_mask += [0] * padding_len
        token_type_ids += [0] * padding_len

        return {
            "input_ids": torch.tensor(token_ids, dtype = torch.long),
            "target_tags": torch.tensor(target_tags, dtype = torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype = torch.long),
            "token_type_ids": torch.tensor(token_type_ids, dtype = torch.long)
        }


The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

**Dataloaders:** Create dataloaders for NER data

In [ ]:
def create_dataloaders(lang_data):

    train_dataset = NERDataset(
        lang_data["train"]["tokens"].to_list(),
        lang_data["train"]["ner_tags"].to_list()
    )
    val_dataset = NERDataset(
        lang_data["val"]["tokens"].to_list(),
        lang_data["val"]["ner_tags"].to_list()
    )
    test_dataset = NERDataset(
        lang_data["test"]["tokens"].to_list(),
        lang_data["test"]["ner_tags"].to_list()
    )

    train_loader = DataLoader(train_dataset, BATCH_SIZE)
    val_loader = DataLoader(val_dataset, BATCH_SIZE)
    test_loader = DataLoader(test_dataset, BATCH_SIZE)

    return train_loader, val_loader, test_loader

**Models:** Create pytorch model architectures

In [ ]:
import torch.nn as nn
from torchcrf import CRF
from transformers import BertModel

class BertBilstmCrf(nn.Module):
    def __init__(self, num_tags):
        super(BertBilstmCrf, self).__init__()

        # Define model layers
        self.bert = BertModel.from_pretrained("bert-base-multilingual-cased")
        self.lstm = nn.LSTM(
            input_size = self.bert.config.hidden_size,
            hidden_size = 128,
            num_layers = 2,
            bidirectional = True,
            batch_first = True,
            dropout = 0.3
        )
        self.fc = nn.Linear(in_features = 256, out_features = num_tags)
        self.crf = CRF(num_tags, batch_first = True)

    def forward(self, input_ids, target_tags, attention_mask, token_type_ids):
        # Pass inputs through layers
        bert_output = self.bert(input_ids, attention_mask, token_type_ids)
        sequence_output = bert_output.last_hidden_state
        lstm_output, _ = self.lstm(sequence_output)
        emissions = self.fc(lstm_output)

        loss = -self.crf(emissions, target_tags, mask = attention_mask.bool(), reduction = "mean")
        return emissions, loss

    def decode(self, emissions, attention_mask):
        return self.crf.decode(emissions, mask = attention_mask.bool())

**Metric Calculation:** Define function to calculate F1-score

In [ ]:
from sklearn.metrics import f1_score

def calculate_f1(target_tags, pred_tags, attention_mask):
    pred_tags = [sequence + [0] * (MAX_SEQ_LEN - len(sequence)) for sequence in pred_tags]
    pred_tags = torch.tensor(pred_tags).to(DEVICE)

    # Flatten batch results
    target_tags = target_tags.view(-1)
    pred_tags = pred_tags.view(-1)
    attention_mask = attention_mask.view(-1)

    # Filter out padding and special tokens
    target_tags = target_tags[attention_mask == 1]
    pred_tags = pred_tags[attention_mask == 1]

    f1_micro = f1_score(target_tags.cpu(), pred_tags.cpu(), average="micro")
    return f1_micro

**Setup Optimizer:** Setup optimizer with different learning rates for seperate layers

In [ ]:
def setup_optimizer(model):
    param_groups = []
    # Check model layers and add appropiate learning rates
    if hasattr(model, "bert"):
        param_groups.append({"params" : model.bert.parameters(), "lr" : BERT_LEARNING_RATE})
    if hasattr(model, "lstm"):
        param_groups.append({"params" : model.lstm.parameters(), "lr" : LSTM_LEARNING_RATE})
    if hasattr(model, "crf"):
        param_groups.append({"params" : model.crf.parameters(), "lr" : CRF_LEARNING_RATE})
    optimizer = optim.Adam(param_groups, weight_decay = WEIGHT_DECAY)

    return optimizer

**Training loop:** Define the loop to train the model

In [ ]:
import copy
import torch.optim as optim

def train_model(model, train_loader, val_loader):
    optimizer = setup_optimizer(model)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode = "min", factor = 0.1, patience = 5)

    best_val_f1 = -float("inf")
    best_train_f1 = 0
    patience_counter = PATIENCE

    for _ in range(EPOCHS):
        train_loss, train_f1 = train_epoch(model, train_loader, optimizer)
        val_loss, val_f1 = evaluate_epoch(model, val_loader)

        scheduler.step(val_loss)

        # Save state of best model
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_train_f1 = train_f1
            patience_counter = PATIENCE
            best_model_state = copy.deepcopy(model.state_dict())
        else:
            patience_counter -= 1

        if patience_counter == 0:
            break  # Stop training if model doesn't improve

    # Delete to clear up memory
    model.to("cpu")
    del optimizer, scheduler, model

    # Clear cache
    gc.collect()
    torch.cuda.empty_cache()

    return best_model_state, best_train_f1, best_val_f1

def train_epoch(model, dataloader, optimizer):
    model.train()
    total_loss, total_f1 = 0, 0

    for batch in dataloader:
        batch = {key : value.to(DEVICE) for key, value in batch.items()}

        optimizer.zero_grad()
        emissions, loss = model(**batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

        pred_tags = model.decode(emissions, batch["attention_mask"])
        f1_score = calculate_f1(batch["target_tags"], pred_tags, batch["attention_mask"])
        total_f1 += f1_score

    return total_loss / len(dataloader), total_f1 / len(dataloader)

def evaluate_epoch(model, dataloader):
    model.eval()
    total_loss, total_f1 = 0, 0

    with torch.no_grad():
        for batch in dataloader:
            batch = {key : value.to(DEVICE) for key, value in batch.items()}

            emissions, loss = model(**batch)
            total_loss += loss.item()

            pred_tags = model.decode(emissions, batch["attention_mask"])
            f1_score = calculate_f1(batch["target_tags"], pred_tags, batch["attention_mask"])
            total_f1 += f1_score

    return total_loss / len(dataloader), total_f1 / len(dataloader)


**Baseline:** Train baseline models; save weights and performance scores for analysis

In [ ]:
from torch.utils.data import DataLoader

baseline_results = {}
print("| Language | Train F1 | Val F1 | Test F1 |")
print("|----------|----------|--------|---------|")

# Iterate through low-resource languages
for lang, lang_data in low_resource_datasets.items():

    train_loader, val_loader, test_loader = create_dataloaders(lang_data)

    # Train
    model = BertBilstmCrf(NUM_TAGS).to(DEVICE)
    best_model_state, train_f1, val_f1 = train_model(model, train_loader, val_loader)

    # Evaluate
    eval_model = BertBilstmCrf(NUM_TAGS).to(DEVICE)
    eval_model.load_state_dict(best_model_state)
    test_loss, test_f1 = evaluate_epoch(eval_model, test_loader)

    # Save results
    baseline_results[lang] = {
        "model_state" : best_model_state,
        "train_f1" : train_f1,
        "val_f1" : val_f1,
        "test_f1" : test_f1
    }

    print(f"| {lang} | {train_f1:.5} | {val_f1:.5} | {test_f1:.5} |")

| Language | Train F1 | Val F1 | Test F1 |
|----------|----------|--------|---------|


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

| mg | 0.99696 | 0.94048 | 0.96476 |
| fo | 0.95772 | 0.91367 | 0.88697 |
| co | 0.98868 | 0.85538 | 0.83268 |
| hsb | 0.99758 | 0.95161 | 0.83662 |
| bh | 0.99543 | 0.89048 | 0.87785 |
| cv | 0.99067 | 0.91779 | 0.87669 |


Baseline Performance

| Language | Train F1 | Val F1 | Test F1 |
|----------|----------|--------|---------|
| mg | 0.99696 | 0.94048 | 0.96476 |
| fo | 0.95772 | 0.91367 | 0.88697 |
| co | 0.98868 | 0.85538 | 0.83268 |
| hsb | 0.99758 | 0.95161 | 0.83662 |
| bh | 0.99543 | 0.89048 | 0.87785 |
| cv | 0.99067 | 0.91779 | 0.87669 |


**Cross-lingual transfer learning:** Train model on high-resource language data, then fine-tune on target low-resource language

In [ ]:
transfer_results = {}
print("| Low-resource Language | High-resource Language | Train F1 | Val F1 | Test F1 |")
print("|-----------------------|------------------------|----------|--------|---------|")

# Iterate through different augmentation factors (ratio of high-resource data to low-resource data)
for aug_factor in range(1, 10 + 1):

    high_resource_datasets = load_wikiann_datasets(high_resource_langs, 300 * aug_factor)
    print(f"Augmentation Factor: {aug_factor}")

    # Iterate through low-resource and adjacent high-resource languages
    for (low_resource_lang, low_resource_data), (high_resource_lang, high_resource_data) in zip(
            low_resource_datasets.items(), high_resource_datasets.items()
        ):

        high_resource_train_loader, high_resource_val_loader, _ = create_dataloaders(high_resource_data)
        low_resource_train_loader, low_resource_val_loader, low_resource_test_loader = create_dataloaders(low_resource_data)


        # Train on high-resource language data
        high_resource_model = BertBilstmCrf(NUM_TAGS).to(DEVICE)
        high_resource_model_state, train_f1, val_f1 = train_model(high_resource_model, high_resource_train_loader, high_resource_val_loader)

        # Fine-tune model on low-resource data
        model = BertBilstmCrf(NUM_TAGS).to(DEVICE)
        model.load_state_dict(high_resource_model_state)
        best_model_state, train_f1, val_f1 = train_model(model, low_resource_train_loader, low_resource_val_loader)

        # Evaluate model on low-resource data
        eval_model = BertBilstmCrf(NUM_TAGS).to(DEVICE)
        eval_model.load_state_dict(best_model_state)
        test_loss, test_f1 = evaluate_epoch(eval_model, low_resource_test_loader)

        # Save transfer learning results
        single_transfer_results = {}
        single_transfer_results[f"{low_resource_lang}_{high_resource_lang}"] = {
            "model_state" : best_model_state,
            "train_f1" : train_f1,
            "val_f1" : val_f1,
            "test_f1" : test_f1
        }
        print(f"| {low_resource_lang} | {high_resource_lang} | {train_f1:.5} | {val_f1:.5} | {test_f1:.5} |")

    transfer_results[aug_factor] = single_transfer_results

Augmentation Factor: 1
Low-resource Language: mg    High-resource Language: id                   Train F1: 0.99334    Val F1: 0.96599    Test F1: 0.97357
Low-resource Language: fo    High-resource Language: is                   Train F1: 0.97982    Val F1: 0.92086    Test F1: 0.90738
Low-resource Language: co    High-resource Language: it                   Train F1: 0.96882    Val F1: 0.84923    Test F1: 0.84825
Low-resource Language: hsb    High-resource Language: pl                   Train F1: 0.98848    Val F1: 0.94556    Test F1: 0.86176
Low-resource Language: bh    High-resource Language: hi                   Train F1: 0.99708    Val F1: 0.89946    Test F1: 0.89251
Low-resource Language: cv    High-resource Language: tt                   Train F1: 0.94709    Val F1: 0.94295    Test F1: 0.83815
Augmentation Factor: 2
Low-resource Language: mg    High-resource Language: id                   Train F1: 0.99576    Val F1: 0.98639    Test F1: 0.96035
Low-resource Language: fo    High-re

Augmentation Factor 1:
| Language | Train F1 | Val F1 | Test F1 |
|----------|---------|--------|--------|
| mg       | 0.99334 | 0.96599 | 0.97357 |
| fo       | 0.97982 | 0.91367 | 0.88540 |
| co       | 0.99179 | 0.87077 | 0.84436 |
| hsb      | 0.99281 | 0.95565 | 0.84740 |
| bh       | 0.84250 | 0.82406 | 0.83713 |
| cv       | 0.94540 | 0.92114 | 0.84008 |

In [ ]:
low_resource_lang = "hf"
high_resource_lang = "hf"
train_f1 = 10.00000001
val_f1 = 10.00000001
test_f1 = 10.00000001


print(f"{low_resource_lang} | {high_resource_lang} | {train_f1:.5} | {val_f1:.5} | {test_f1:.5}")


hf | hf | 10.0 | 10.0 | 10.0
